In [1]:
from utils import setrootdir
setrootdir("ppgcc-coautorias")

'Directory ppgcc-coautorias successfully loaded as current working directory.'

In [2]:
import os

from dotenv import load_dotenv

import seaborn as sns

sns.set_style("whitegrid")

from src.transformation import Transformer

In [3]:
load_dotenv()

DATASET_DIRECTORY = os.getenv("DATASET_DIRECTORY")
METADATA_FILE = os.getenv("METADATA_FILE")

# 3. Transformation

In [4]:
transformer = Transformer(
    data_dir=DATASET_DIRECTORY,
    metadata_file=METADATA_FILE,
    step_directory="03-transformation"
)

In [5]:
df_productions = transformer.read_parquet(step="02-preprocessing", name="productions")
df_productions

,name,citation,lattes_id,institution,production,authors,location,type,year,issn,production_id
0,Jose Maria Nazar David,"[Jose Maria Nazar David, DAVID, JOSE MARIA, DA...",3640497501056163,UFJF,SCIENTIFIC PROVENANCE METADATA CAPTURE AND MAN...,Wander Gaspar,"International Journal of Metadata, Semantics a...",PERIODICO,2015,17442621,129626
1,Jose Maria Nazar David,"[Jose Maria Nazar David, DAVID, JOSE MARIA, DA...",3640497501056163,UFJF,SCIENTIFIC PROVENANCE METADATA CAPTURE AND MAN...,Regina Maria Maciel Braga,"International Journal of Metadata, Semantics a...",PERIODICO,2015,17442621,129626
2,Jose Maria Nazar David,"[Jose Maria Nazar David, DAVID, JOSE MARIA, DA...",3640497501056163,UFJF,SCIENTIFIC PROVENANCE METADATA CAPTURE AND MAN...,Fernanda Cláudia Alves Campos,"International Journal of Metadata, Semantics a...",PERIODICO,2015,17442621,129626
3,Jose Maria Nazar David,"[Jose Maria Nazar David, DAVID, JOSE MARIA, DA...",3640497501056163,UFJF,SCIENTIFIC PROVENANCE METADATA CAPTURE AND MAN...,Jose Maria Nazar David,"International Journal of Metadata, Semantics a...",PERIODICO,2015,17442621,129626
4,Jose Maria Nazar David,"[Jose Maria Nazar David, DAVID, JOSE MARIA, DA...",3640497501056163,UFJF,SCIENTIFIC PROVENANCE METADATA CAPTURE AND MAN...,Tatiane Ornelas Martins Alves,"International Journal of Metadata, Semantics a...",PERIODICO,2015,17442621,129626
...,...,...,...,...,...,...,...,...,...,...,...
495575,Leandro Santiago de Araujo,"[Leandro Santiago de Araujo, L. S. ARAUJO, LEA...",6358983442870515,UFF,GERANDO BASE DE GRAFOS NÃO ISOMORFOS COM SEUS ...,Luís Felipe Ignácio Cunha,SBPO 2023: Simpósio Brasileiro de Pesquisa Ope...,CONFERENCIA,2023,None,76459
495576,Leandro Santiago de Araujo,"[Leandro Santiago de Araujo, L. S. ARAUJO, LEA...",6358983442870515,UFF,PARALELISMO E HEURÍSTICAS PARA O PROBLEMA DA M...,Thiago Lopes Nascimento,SBPO 2023: Simpósio Brasileiro de Pesquisa Ope...,CONFERENCIA,2023,None,112702
495577,Leandro Santiago de Araujo,"[Leandro Santiago de Araujo, L. S. ARAUJO, LEA...",6358983442870515,UFF,PARALELISMO E HEURÍSTICAS PARA O PROBLEMA DA M...,Fábio Protti,SBPO 2023: Simpósio Brasileiro de Pesquisa Ope...,CONFERENCIA,2023,None,112702
495578,Leandro Santiago de Araujo,"[Leandro Santiago de Araujo, L. S. ARAUJO, LEA...",6358983442870515,UFF,PARALELISMO E HEURÍSTICAS PARA O PROBLEMA DA M...,Luís Felipe Ignácio Cunha,SBPO 2023: Simpósio Brasileiro de Pesquisa Ope...,CONFERENCIA,2023,None,112702


## 3.1. Unique IDs for names and authors

In [6]:
unique_names = set(df_productions["name"].unique())
len(unique_names)

1756

In [7]:
unique_authors = set(df_productions["authors"].unique())
unique_authors = unique_authors.difference(unique_names)
len(unique_authors)

121341

In [8]:
unique_names = sorted(list(unique_names))
unique_authors = sorted(list(unique_authors))
unique_names = unique_names + unique_authors
len(unique_names)

123097

In [9]:
# map_id = transformer.map_name_id(unique_names)
# len(map_id)

i = 0

map_id = {}

for name in unique_names:
    map_id[name] = i
    i += 1

len(map_id)

123097

In [10]:
df_productions["nid"] = transformer.set_id_column(df_productions["name"], map_id)
df_productions["aid"] = transformer.set_id_column(df_productions["authors"], map_id)

df_productions = df_productions[["production_id", "name", "citation", "nid", "lattes_id", "institution","production", "aid", "authors", "type", "year", "issn"]]
df_productions

,production_id,name,citation,nid,lattes_id,institution,production,aid,authors,type,year,issn
0,129626,Jose Maria Nazar David,"[Jose Maria Nazar David, DAVID, JOSE MARIA, DA...",830,3640497501056163,UFJF,SCIENTIFIC PROVENANCE METADATA CAPTURE AND MAN...,119076,Wander Gaspar,PERIODICO,2015,17442621
1,129626,Jose Maria Nazar David,"[Jose Maria Nazar David, DAVID, JOSE MARIA, DA...",830,3640497501056163,UFJF,SCIENTIFIC PROVENANCE METADATA CAPTURE AND MAN...,1384,Regina Maria Maciel Braga,PERIODICO,2015,17442621
2,129626,Jose Maria Nazar David,"[Jose Maria Nazar David, DAVID, JOSE MARIA, DA...",830,3640497501056163,UFJF,SCIENTIFIC PROVENANCE METADATA CAPTURE AND MAN...,43349,Fernanda Cláudia Alves Campos,PERIODICO,2015,17442621
3,129626,Jose Maria Nazar David,"[Jose Maria Nazar David, DAVID, JOSE MARIA, DA...",830,3640497501056163,UFJF,SCIENTIFIC PROVENANCE METADATA CAPTURE AND MAN...,830,Jose Maria Nazar David,PERIODICO,2015,17442621
4,129626,Jose Maria Nazar David,"[Jose Maria Nazar David, DAVID, JOSE MARIA, DA...",830,3640497501056163,UFJF,SCIENTIFIC PROVENANCE METADATA CAPTURE AND MAN...,112697,Tatiane Ornelas Martins Alves,PERIODICO,2015,17442621
...,...,...,...,...,...,...,...,...,...,...,...,...
495575,76459,Leandro Santiago de Araujo,"[Leandro Santiago de Araujo, L. S. ARAUJO, LEA...",952,6358983442870515,UFF,GERANDO BASE DE GRAFOS NÃO ISOMORFOS COM SEUS ...,1050,Luís Felipe Ignácio Cunha,CONFERENCIA,2023,None
495576,112702,Leandro Santiago de Araujo,"[Leandro Santiago de Araujo, L. S. ARAUJO, LEA...",952,6358983442870515,UFF,PARALELISMO E HEURÍSTICAS PARA O PROBLEMA DA M...,113452,Thiago Lopes Nascimento,CONFERENCIA,2023,None
495577,112702,Leandro Santiago de Araujo,"[Leandro Santiago de Araujo, L. S. ARAUJO, LEA...",952,6358983442870515,UFF,PARALELISMO E HEURÍSTICAS PARA O PROBLEMA DA M...,642,Fábio Protti,CONFERENCIA,2023,None
495578,112702,Leandro Santiago de Araujo,"[Leandro Santiago de Araujo, L. S. ARAUJO, LEA...",952,6358983442870515,UFF,PARALELISMO E HEURÍSTICAS PARA O PROBLEMA DA M...,1050,Luís Felipe Ignácio Cunha,CONFERENCIA,2023,None


In [11]:
flag_productions = {}

for index, row in df_productions.iterrows():
    name = row["nid"]
    authors = row["aid"]

    if name == authors:
        continue

    production = row["production_id"]

    if production not in flag_productions:
        flag_productions[production] = {}

    pair = frozenset([name, authors])

    if pair not in flag_productions[production]:
        flag_productions[production][pair] = False

In [12]:
transformer.write_parquet(df_productions, step="03-transformation", name="productions")

In [13]:
import pandas as pd
import numpy as np

def frame_coauthorship_adjacency_matrix(
        df: pd.DataFrame, start_year: int = None, end_year: int = None,
        institution: str = None, remove_isolated: bool = False
        ) -> pd.DataFrame:

    if start_year is not None and end_year is not None:
        if start_year == end_year:
            df_filtered = df[df["year"] == start_year].copy()
    else:
        if start_year is None:
            start_year = df["year"].min()
        if end_year is None:
            end_year = df["year"].max()

    nunique = df["nid"].nunique()

    nids = np.sort(df["nid"].unique())

    unique_names = sorted(list(df["name"].unique()))

    data_adjacency = np.zeros(shape=(nunique, nunique), dtype=np.uint32)

    df_filtered = df[(df["year"] >= start_year) & (df["year"] <= end_year)].copy()

    if institution is not None:
        df_filtered = df_filtered[df_filtered["institution"] == institution]

    for _, row in df_filtered.iterrows():
        researcher_name = row["name"]
        researcher_id = row["nid"]
        citation_name = row["authors"]
        citation_id = row["aid"]
        production_id = row["production_id"]

        if citation_name in unique_names and citation_name != researcher_name:

            if not flag_productions[production_id][frozenset([researcher_id, citation_id])]:
                flag_productions[production_id][frozenset([researcher_id, citation_id])] = True

                i = researcher_id
                j = citation_id

                try:
                    data_adjacency[i][j] += 1
                    data_adjacency[j][i] += 1
                except IndexError:
                    continue

    df_adjacency = pd.DataFrame(data_adjacency, columns=nids, index=nids)

    if remove_isolated:
        df_adjacency = df_adjacency[df_adjacency.sum() > 0]
        df_adjacency = df_adjacency[df_adjacency.index]

    return df_adjacency

In [14]:
df_adjacency = frame_coauthorship_adjacency_matrix(
    df=df_productions,
    remove_isolated=True
)

df_adjacency

,0,1,2,3,4,5,6,7,8,9,...,1746,1747,1748,1749,1750,1751,1752,1753,1754,1755
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1751,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1752,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
1753,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1754,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0


In [16]:
max_col = df_adjacency.max().idxmax()
max_row = df_adjacency[max_col].idxmax()

max_col, max_row, df_adjacency[max_col][max_row]

(np.int64(341), np.int64(1497), np.uint32(224))

In [19]:
transformer.write_parquet(df_adjacency, step="03-transformation", name="adjacency")

In [20]:
for year in range(2014, 2023):

    df_adjacency_yearly = transformer.frame_coauthorship_adjacency_matrix(
        df_productions,
        start_year=year,
        end_year=year,
        remove_isolated=True
    )

    transformer.write_parquet(df_adjacency_yearly, step="03-transformation", name=f"adjacency_{year}")